# 11. 프롬프트와 OCR, 해상도 비교

담당: 창혁
목표: 프롬프트 두 가지와 max_pixels 후보를 고정 표본에서 비교한 뒤, 팀 전체가 사용할 입력 설정을
하나로 동결하는 것입니다.

계획서 5절의 원칙을 그대로 따릅니다. 비교는 이미지만 사용하는 방식과 OCR을 보조로 붙이는 방식,
두 가지로만 제한합니다. 후보를 늘리면 시간이 부족해지고 표본에 과적합될 위험이 커집니다.

## 선행 조건
수빈의 01번 노트북이 끝나서 `artifacts/data/split_v1.json` 이 존재해야 합니다. 개발 표본은 반드시
train ID에서만 뽑습니다. validation에서 뽑으면 이후의 모든 정확도 비교가 오염됩니다.

## 통과 기준(PASS)
1. 대표 이미지 20장 정도를 실제로 눈으로 확인했습니다.
2. 고정 표본 128문항에서 조합별 정확도와 소요 시간을 표로 확인했습니다.
3. `artifacts/input_config_frozen.json` 을 기록했고, 그 내용을 준혁과 유림에게 전달했습니다.

## 넘겨줄 내용
동결한 prompt_id, OCR 사용 여부, 모델별 max_pixels 값입니다. 학습이 시작된 뒤에는 이 값을
바꾸지 않습니다.

### 명령 인자에 대한 안내

이 노트북의 명령은 `vqa/` 와 `scripts/` 에 실제로 구현된 인자에 맞춰 작성했습니다. 그래도 코어
코드가 계속 바뀌는 동안에는 인자 이름이 달라질 수 있습니다. 셀이 `unrecognized arguments` 로
실패하면 먼저 해당 명령에 `--help` 를 붙여서 실제 인자 이름을 확인하고, 셀 안의 문자열만 고치면
됩니다. 노트북의 다른 부분은 고치지 않아도 됩니다. 고친 내용은 팀 채널에 알려서 다른 사람이 같은
곳에서 막히지 않게 하십시오.

In [ ]:
# ===== 파라미터 =====
PROJECT_ROOT = r"."
DATA_ROOT = r"."
RUN_ROOT = r"./artifacts/runs"
LOCAL_SCRATCH = r"./artifacts/scratch"
MODEL_ROOT = r""

SPLIT_FILE = "artifacts/data/split_v1.json"
DEV_SAMPLE_FILE = "artifacts/data/prompt_dev_128.txt"
DEV_SAMPLE_N = 128
SAMPLE_SEED = 20260921

CONFIG_MAIN = "configs/lab_qwen35_9b.yaml"      # 9B가 준비되면 이것을 사용합니다.
CONFIG_FALLBACK = "configs/lab_qwen25vl_3b.yaml"  # 9B가 아직 안 되면 3B로 대신합니다.
USE_FALLBACK = False

RUN_ID = "lab_prompt_cmp_v1"
PROMPT_IDS = ["v1_image_only", "v1_ocr"]
# config의 max_pixels는 "시각 토큰 수 x 패치 넓이"로 적습니다. Qwen3.5는 패치가 32x32이고
# Qwen2.5-VL은 28x28이므로, 후보는 토큰 수로 적고 아래에서 환산합니다.
VISUAL_TOKEN_CANDIDATES = [768, 512]
PATCH_AREA = {"lab_qwen35_9b": 32 * 32, "lab_qwen25vl_3b": 28 * 28}

IMAGE_SAMPLE_N = 20
KEYWORDS = ["영수증", "메뉴", "전화", "번호", "간판", "안내", "표지", "가격", "시간", "주소"]

In [ ]:
# --- 공통 부트스트랩: 경로, 환경 변수, 명령 실행 헬퍼 (수정할 일 거의 없음) ---
import os, sys, json, shlex, subprocess, time, csv, hashlib, random
from pathlib import Path

PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


def _setenv(name, value, create=False):
    if not value:
        return None
    p = Path(value).expanduser()
    if create:
        p.mkdir(parents=True, exist_ok=True)
    os.environ[name] = str(p)
    return p


DATA_ROOT_P = _setenv("VQA_DATA_ROOT", DATA_ROOT)
RUN_ROOT_P = _setenv("VQA_RUN_ROOT", RUN_ROOT, create=True)
SCRATCH_P = _setenv("VQA_LOCAL_SCRATCH", LOCAL_SCRATCH, create=True)
MODEL_ROOT_P = _setenv("VQA_MODEL_ROOT", MODEL_ROOT)

PY = sys.executable


def run(cmd, check=True, echo=True, quiet=False):
    """명령을 실행하면서 출력을 그대로 보여 주고 종료 코드와 소요 시간을 돌려줍니다."""
    if isinstance(cmd, str):
        cmd = shlex.split(cmd)
    cmd = [str(x) for x in cmd]
    if echo:
        print("$ " + " ".join(shlex.quote(x) for x in cmd), flush=True)
    t0 = time.time()
    proc = subprocess.Popen(
        cmd, cwd=str(PROJECT_ROOT), stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, encoding="utf-8",
        errors="replace", bufsize=1,
    )
    buf = []
    for line in proc.stdout:
        buf.append(line)
        if not quiet:
            print(line, end="", flush=True)
    rc = proc.wait()
    dt = time.time() - t0
    print("[rc=%d] %.1f초" % (rc, dt), flush=True)
    if rc != 0:
        print("실패했습니다. 같은 명령에 --help 를 붙여서 인자 이름부터 확인하십시오.", flush=True)
        if check:
            raise RuntimeError("명령이 실패했습니다: " + " ".join(cmd))
    return {"rc": rc, "seconds": dt, "output": "".join(buf)}


def vqa(module, *args, **kw):
    """python -m vqa.<module> 을 실행합니다."""
    return run([PY, "-m", "vqa." + module] + [str(a) for a in args], **kw)


def script(name, *args, **kw):
    """scripts/<name> 을 실행합니다."""
    return run([PY, str(Path("scripts") / name)] + [str(a) for a in args], **kw)


def read_csv_rows(path):
    with open(path, newline="", encoding="utf-8") as fh:
        return list(csv.DictReader(fh))


def read_ids(path):
    return [ln.strip() for ln in Path(path).read_text(encoding="utf-8").splitlines() if ln.strip()]


def sha256_of(path, limit=None):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    digest = h.hexdigest()
    return digest[:limit] if limit else digest


print("PROJECT_ROOT      :", PROJECT_ROOT)
for _name in ("VQA_DATA_ROOT", "VQA_RUN_ROOT", "VQA_LOCAL_SCRATCH", "VQA_MODEL_ROOT"):
    print("%-18s: %s" % (_name, os.environ.get(_name, "(미설정)")))

In [ ]:
# --- 이미지 미리보기 헬퍼 (표시 전용) ---
import base64, io
from IPython.display import HTML, display
from PIL import Image

Image.MAX_IMAGE_PIXELS = None


def thumb_b64(path, size=380):
    im = Image.open(path).convert("RGB")
    im.thumbnail((size, size))
    buf = io.BytesIO()
    im.save(buf, format="JPEG", quality=80)
    return base64.b64encode(buf.getvalue()).decode()


def show_images(paths, captions=None, size=380):
    captions = captions or [Path(p).name for p in paths]
    html = ['<div style="display:flex;gap:10px;flex-wrap:wrap;align-items:flex-start">']
    for p, cap in zip(paths, captions):
        html.append(
            '<figure style="margin:0;font-size:12px">'
            '<img src="data:image/jpeg;base64,%s" style="max-width:%dpx">'
            "<figcaption>%s</figcaption></figure>" % (thumb_b64(p, size), size, cap)
        )
    html.append("</div>")
    display(HTML("".join(html)))


print("이미지 미리보기 헬퍼를 준비했습니다.")

In [ ]:
CONFIG = CONFIG_FALLBACK if USE_FALLBACK else CONFIG_MAIN
CONFIG_KEY = Path(CONFIG).stem
AREA = PATCH_AREA[CONFIG_KEY]
MAX_PIXELS_CANDIDATES = [tok * AREA for tok in VISUAL_TOKEN_CANDIDATES]
print("사용할 config:", CONFIG, "(존재:", Path(CONFIG).exists(), ")")
print("패치 넓이 %d, max_pixels 후보 %s" % (AREA, MAX_PIXELS_CANDIDATES))

if not Path(SPLIT_FILE).exists():
    raise FileNotFoundError(
        SPLIT_FILE + " 이 아직 없습니다. 수빈의 01번 노트북이 끝날 때까지 기다려야 합니다."
    )
SPLIT = json.loads(Path(SPLIT_FILE).read_text(encoding="utf-8"))
TRAIN_IDS = list(SPLIT["train_ids"])
VAL_IDS = set(SPLIT["val_ids"])
print("train %d개 / val %d개" % (len(TRAIN_IDS), len(VAL_IDS)))

## 1. 대표 이미지 20장 확인

작은 글자, 영수증, 메뉴판, 전화번호처럼 판독이 어려운 사례를 우선 고릅니다. 키워드로 1차 후보를
뽑은 다음, 부족하면 무작위로 채웁니다. 이 단계의 목적은 어떤 해상도에서 글자가 읽히는지를
사람이 직접 판단하는 것입니다.

In [ ]:
ROWS = read_csv_rows(Path(os.environ["VQA_DATA_ROOT"]) / "train.csv")
ID2ROW = {r["id"]: r for r in ROWS}

def row_text(r):
    return " ".join(str(r.get(k, "")) for k in ("question", "a", "b", "c", "d"))

keyword_hits = [r for r in ROWS if any(k in row_text(r) for k in KEYWORDS)]
rng = random.Random(SAMPLE_SEED)
rng.shuffle(keyword_hits)
picked = keyword_hits[:IMAGE_SAMPLE_N]
if len(picked) < IMAGE_SAMPLE_N:
    rest = [r for r in ROWS if r not in picked]
    rng.shuffle(rest)
    picked += rest[: IMAGE_SAMPLE_N - len(picked)]

print("대표 이미지 %d장을 골랐습니다." % len(picked))
REPRESENTATIVE_IDS = [r["id"] for r in picked]
Path("artifacts/data").mkdir(parents=True, exist_ok=True)
Path("artifacts/data/representative_ids.txt").write_text(
    "\n".join(REPRESENTATIVE_IDS) + "\n", encoding="utf-8")

for r in picked:
    print("[%s] %s" % (r["id"], r["question"][:60]))
show_images([Path(os.environ["VQA_DATA_ROOT"]) / r["path"] for r in picked],
            captions=[r["id"] for r in picked], size=300)

## 2. 고정 개발 표본 128문항 만들기

반드시 train ID에서만 뽑습니다. 아래 셀은 val과 겹치지 않는지 직접 확인하고, 겹치면 멈춥니다.
표본은 seed로 고정되므로 누가 언제 실행해도 같은 문항이 나옵니다.

In [ ]:
rng2 = random.Random(SAMPLE_SEED)
pool = sorted(set(TRAIN_IDS) - VAL_IDS)
sample_ids = sorted(rng2.sample(pool, min(DEV_SAMPLE_N, len(pool))))

leak = set(sample_ids) & VAL_IDS
assert not leak, "개발 표본에 val ID가 섞였습니다: %s" % sorted(leak)[:5]

Path(DEV_SAMPLE_FILE).write_text("\n".join(sample_ids) + "\n", encoding="utf-8")
print("개발 표본 %d문항을 저장했습니다: %s" % (len(sample_ids), DEV_SAMPLE_FILE))
print("val과 겹치는 ID: 0개")
print("표본 sha256:", sha256_of(DEV_SAMPLE_FILE)[:16])

## 3. 조합별 비교 실행

프롬프트 2종과 max_pixels 후보를 조합해서 순서대로 돌립니다. 조합이 늘어나면 시간이 비례해서
늘어나므로, 후보는 최소한으로 유지하십시오. 도중에 멈춰도 다시 실행하면 이미 끝난 조합의 예측은
건너뜁니다.

In [ ]:
COMBOS = [(p, mp) for p in PROMPT_IDS for mp in MAX_PIXELS_CANDIDATES]
RESULTS = []

for prompt_id, max_pixels in COMBOS:
    tokens = max_pixels // AREA
    name = "cmp_%s_%dtok" % (prompt_id, tokens)
    print("=" * 70)
    print("조합: prompt_id=%s / max_pixels=%d (시각 토큰 %d개)" % (prompt_id, max_pixels, tokens))
    res = vqa("predict",
              "--config", CONFIG,
              "--run-id", RUN_ID,
              "--csv", "train.csv",
              "--ids-file", DEV_SAMPLE_FILE,
              "--adapter", "none",
              "--set", "prompt_id=%s" % prompt_id,
              "--set", "max_pixels=%d" % max_pixels,
              "--out-name", name,
              check=False)
    RESULTS.append({"prompt_id": prompt_id, "max_pixels": max_pixels, "tokens": tokens,
                    "out_name": name, "seconds": res["seconds"], "rc": res["rc"]})

In [ ]:
Path("artifacts/eval").mkdir(parents=True, exist_ok=True)
PRED_DIR = RUN_ROOT_P / RUN_ID / "preds"

for row in RESULTS:
    eval_out = "artifacts/eval/%s_%s.json" % (RUN_ID, row["out_name"])
    r = script("evaluate_run.py",
               "--pred", str(PRED_DIR / (row["out_name"] + ".csv")),
               "--gold", str(Path(os.environ["VQA_DATA_ROOT"]) / "train.csv"),
               "--val-ids", DEV_SAMPLE_FILE,
               "--json-out", eval_out,
               "--no-breakdown", "--no-log",
               check=False, quiet=True)
    acc = None
    if Path(eval_out).exists():
        data = json.loads(Path(eval_out).read_text(encoding="utf-8"))
        for key in ("accuracy", "acc", "top1"):
            if key in data:
                acc = data[key]
                break
    row["accuracy"] = acc

## 4. 비교 표

정확도 차이가 128문항에서 1퍼센트에서 2퍼센트 수준이라면 표본 오차 범위 안입니다. 그 경우에는
더 단순하고 빠른 쪽을 고르는 편이 낫습니다. OCR을 채택하면 학습과 validation, test 전부에서 같은
입력 규칙을 써야 하고 OCR 캐시도 준비해야 하므로, 뚜렷한 이득이 있을 때만 선택하십시오.

In [ ]:
print("%-16s %-12s %-10s %-10s" % ("prompt_id", "시각토큰", "정확도", "소요(초)"))
print("-" * 54)
for row in sorted(RESULTS, key=lambda r: (-(r["accuracy"] or 0), r["seconds"])):
    acc_txt = "%.3f" % row["accuracy"] if row["accuracy"] is not None else "미산출"
    print("%-16s %-12s %-10s %-10.1f"
          % (row["prompt_id"], "%d개" % row["tokens"], acc_txt, row["seconds"]))
print()
print("표본은 %d문항이므로 작은 차이는 우연일 수 있습니다." % DEV_SAMPLE_N)

## 5. 입력 설정 동결

아래 값을 직접 정해서 적은 뒤 셀을 실행합니다. 이 파일이 저장되는 순간부터 팀 전체의 입력 규칙이
고정됩니다. 준혁의 27B 실행과 유림의 9B 학습이 이 파일을 보고 설정을 맞춥니다.

In [ ]:
# ===== 여기에 최종 결정을 적습니다 =====
FROZEN_PROMPT_ID = "v1_image_only"
FROZEN_OCR_ENABLED = False
# config와 같은 규칙으로 적습니다. 시각 토큰 수에 패치 넓이를 곱한 값입니다.
FROZEN_MAX_PIXELS = {
    "lab_qwen35_9b": 768 * 32 * 32,
    "lab_qwen25vl_3b": 768 * 28 * 28,
    "colab_qwen35_27b": 1024 * 32 * 32,
}
DECIDED_BY = "창혁"
NOTE = "128문항 비교 결과 OCR의 이득이 뚜렷하지 않아 이미지 단독으로 시작합니다."

frozen = {
    "prompt_id": FROZEN_PROMPT_ID,
    "ocr_enabled": FROZEN_OCR_ENABLED,
    "max_pixels": FROZEN_MAX_PIXELS,
    "decided_by": DECIDED_BY,
    "note": NOTE,
    "dev_sample_file": DEV_SAMPLE_FILE,
    "dev_sample_sha256": sha256_of(DEV_SAMPLE_FILE),
    "split_file": SPLIT_FILE,
    "comparison": RESULTS,
    "frozen_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}
out = Path("artifacts/input_config_frozen.json")
out.write_text(json.dumps(frozen, ensure_ascii=False, indent=2), encoding="utf-8")

print("===== 입력 설정 동결 =====")
print("파일       : %s" % out)
print("prompt_id  : %s" % FROZEN_PROMPT_ID)
print("OCR 사용   : %s" % ("예" if FROZEN_OCR_ENABLED else "아니오"))
for k, v in FROZEN_MAX_PIXELS.items():
    area = PATCH_AREA.get(k, 32 * 32)
    print("max_pixels : %-22s %d (시각 토큰 %d개)" % (k, v, v // area))
print("받는 사람  : 준혁(27B 실행), 유림(9B 학습)")
print("주의       : 학습이 시작된 뒤에는 이 값을 바꾸지 않습니다.")
print("=========================")